# Notebook 04: Multi-Objective Optimization Solver (NSGA-II & MILP)

**Objective:** Formulate and solve the Capacitated Multi-Objective EV Charging Station Location and Sizing Problem (CMO-EVCSLSP) using NSGA-II to extract non-dominated Pareto-optimal station placement vectors $(x, Y)$.

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.optimization.nsga2_solver import NSGA2Solver, export_pareto_solutions
from src.visualization.pareto_front import plot_pareto_front_2d, find_knee_point_solution

config_path = os.path.join(module_path, 'configs', 'dhaka_scenario_2030.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

od_path = os.path.join(module_path, 'data', 'processed', 'od_travel_time_matrix.npz')
cand_path = os.path.join(module_path, 'data', 'processed', 'candidate_sites_filtered.geojson')

data = np.load(od_path)
dist_matrix = data['distances']
time_matrix = data['travel_times']
demand_values = data['demand_values']

cand_gdf = gpd.read_file(cand_path)
cand_meta = cand_gdf.to_dict(orient='records')

solver = NSGA2Solver(dist_matrix, time_matrix, demand_values, cand_meta, config)
solver.pop_size = 50
solver.generations = 40

print('Executing NSGA-II Solver (Population: 50, Generations: 40)...')
pareto_front, population = solver.solve(generations=40)

output_csv = os.path.join(module_path, 'results', 'tables', 'optimal_solutions_pareto.csv')
pareto_df = export_pareto_solutions(pareto_front, cand_meta, config, output_csv)
display(pareto_df.head(10))

## 1. Pareto Trade-Off Curve Analysis & Knee-Point Selection

In [ ]:
fig_path = os.path.join(module_path, 'results', 'figures', 'pareto_frontier_tradeoff.png')
plot_pareto_front_2d(pareto_df, fig_path)

knee_idx, knee_sol = find_knee_point_solution(pareto_df)
print(f"Selected Knee-Point Solution: {knee_sol['solution_id']}")
print(f"  - Total System Cost: BDT {knee_sol['total_cost_million_bdt']:.2f} Million")
print(f"  - Demand Coverage: {knee_sol['demand_coverage_pct']:.1f}%")
print(f"  - Deployed Stations: {knee_sol['open_station_count']} sites")
print(f"  - Total Grid Demand: {knee_sol['total_grid_power_kw']:.1f} kW")